# HDS Patient Outreach Analytics Data Validation Tests

In [ ]:
workspace_id = ""
destination_lakehouse_id = ""
destination_lakehouse_path = ""
deployment_environment = ""
silver_lakehouse_id=""
poagold_lakehouse_id=""

In [ ]:
from pyspark.sql.functions import input_file_name, col, count, countDistinct, coalesce, lit
from pyspark.sql import DataFrame
from pyspark.sql.types import StructType, StructField, StringType, LongType
from concurrent.futures import ThreadPoolExecutor
import unittest
from pyspark.sql import SparkSession
import io
import logging
import sempy.fabric as fabric
import xmlrunner

class PatientOutreachAnalyticsDataValidationTests(unittest.TestCase):

    def __init__(self, methodName='runTest', spark=None, workspace_id = None, bronze_lakehouse_id = None, databases = []):
        super().__init__(methodName)
        logging.basicConfig()
        self.logger = logging.getLogger("LOG")
        self.spark = spark
        self.workspace_id = workspace_id
        self.poagold_lakehouse_id = poagold_lakehouse_id
        self.databases = databases
        self.silver_lakehouse_id=silver_lakehouse_id

    # TODO - Add data validation tests
    def test_poa_bronze_marketing_tables_contains_data(self):
        table_names = ['contact_partitioned','msdynmkt_email_partitioned','msdynmkt_journey_partitioned','msdynmkt_journeydependency_partitioned','msdynmkt_journeyinstance_partitioned','msdynmkt_pushnotification_partitioned','msdynmkt_sms_partitioned','msemr_codeableconcept_partitioned']
        for table_name in table_names:
            count_df = spark.sql(f"SELECT COUNT(*) AS count FROM {table_name}")
            count = count_df.collect()[0]["count"]
            self.assertGreater(count, 0, "Each POA marketing table should have non-zero counts.")
            
    def test_poa_bronze_marketing_analytics_tables_contains_data(self):
        table_names = ['ActionEventInflow','JourneyEventEntryProcessed','JourneyEventInflow','EmailSent','EmailOpened','EmailDelivered','SmsNotSent']
        for table_name in table_names:
            count_df = spark.sql(f"SELECT COUNT(*) AS count FROM {table_name}")
            count = count_df.collect()[0]["count"]
            self.assertGreater(count, 0, "Each POA marketing analytics table should have non-zero counts.")
            
    def test_poa_silver_tables_contains_data(self):
        table_names = ['ApplicationCommunication','CommunicationMethod','CommunicationStatus','CommunicationStatusType','EmailCommunication','IdmCommunication','IdmPatient','Language','MarketingCampaignTask','MarketingCampaignTaskCommunication','MarketingCampaignTaskRelatedParty','MarketingCampaignTaskStatus','MarketingCampaignTaskType','Party','PartyType','RelatedCommunication','RelatedMarketingCampaignTask','SmsCommunication','SoftwareProduct','SoftwareProductVersion','TaskPartyRelationshipType','TaskRelationshipType','TaskStatusType']
        for table_name in table_names:
            count_df = spark.sql(f"SELECT COUNT(*) AS count FROM `{self.silver_lakehouse_id}`.{table_name}")
            count = count_df.collect()[0]["count"]
            self.assertGreater(count, 0, "Each POA silver table should have non-zero counts.")
            
    def test_poa_compare_source_target_data_patient(self):
        source_query="""select count(*) as count from contact_partitioned"""
        target_query=f"""select count(*) as count from `{self.silver_lakehouse_id}`.IdmPatient"""
        source_df=spark.sql(source_query)
        target_df=spark.sql(target_query)
        source_count= source_df.collect()[0]["count"]
        target_count= target_df.collect()[0]["count"]
        self.assertEqual(source_count, target_count)
        
    def test_poa_compare_source_target_data_journey(self):
        source_query="""select count(*) as count from msdynmkt_journey_partitioned j left join msemr_codeableconcept_partitioned c on j.msemr_serviceline = c.msemr_codeableconceptid where j.msdynmkt_journeyid!= 'NULL'"""
        target_query=f"""SELECT DISTINCT count(*) as count FROM `{self.silver_lakehouse_id}`.MarketingCampaignTask AS mct
                        LEFT JOIN ( SELECT rmct.TaskId,rmct.RelatedTaskId FROM `{self.silver_lakehouse_id}`.RelatedMarketingCampaignTask rmct WHERE rmct.SourceTable = 'msdynmkt_journey_partitioned_IDM') rmctj 
                        ON mct.TaskId = rmctj.TaskId
                        LEFT JOIN `{self.silver_lakehouse_id}`.MarketingCampaignTaskStatus AS mcts 
                        ON mct.TaskId = mcts.TaskId
                        LEFT JOIN `{self.silver_lakehouse_id}`.TaskStatusType AS tst 
                        ON mcts.TaskStatusTypeId = tst.TaskStatusTypeId
                        WHERE mct.MarketingCampaignTaskTypeId = 20000001 """
        
        source_df=spark.sql(source_query)
        target_df=spark.sql(target_query)
        source_count= source_df.collect()[0]["count"]
        target_count= target_df.collect()[0]["count"]
        self.assertEqual(source_count, target_count)
        
    def test_poa_compare_source_target_data_actionevent(self):
        source_query="""Select count(*) as count FROM actioneventinflow"""
        target_query=f"""SELECT  count(*) as count FROM `{self.silver_lakehouse_id}`.MarketingCampaignTask mct 
                JOIN (SELECT rmct.TaskId, rmct.RelatedTaskId FROM `{self.silver_lakehouse_id}`.RelatedMarketingCampaignTask rmct 
                WHERE rmct.SourceTable = 'actioneventinflow_IDM') rmctj 
                ON mct.TaskId = rmctj.TaskId JOIN `{self.silver_lakehouse_id}`.MarketingCampaignTaskStatus mcts 
                ON mct.TaskId = mcts.TaskId JOIN `{self.silver_lakehouse_id}`.TaskStatusType tst 
                ON mcts.TaskStatusTypeId = tst.TaskStatusTypeId 
                JOIN (SELECT mctrp.PartyId, mctrp.TaskId FROM `{self.silver_lakehouse_id}`.MarketingCampaignTaskRelatedParty mctrp 
                WHERE SourceTable = 'actioneventinflow_IDM') mctrpt ON mct.TaskId = mctrpt.TaskId 
                JOIN `{self.silver_lakehouse_id}`.Party pt ON mctrpt.PartyId = pt.PartyId 
                JOIN (SELECT rmct.TaskId, rmct.RelatedTaskId FROM `{self.silver_lakehouse_id}`.RelatedMarketingCampaignTask rmct WHERE rmct.SourceTable = 'actioneventinflow_relatedjourneyinstance_IDM') rmctji 
                ON mct.TaskId = rmctji.TaskId JOIN (SELECT rmct.TaskId, rmct.RelatedTaskId 
                FROM `{self.silver_lakehouse_id}`.RelatedMarketingCampaignTask rmct WHERE rmct.SourceTable = 'actioneventinflow_relatedjourneyaction_IDM') rmctja 
                ON mct.TaskId = rmctja.TaskId JOIN (SELECT mctrp.PartyId, mctrp.TaskId FROM `{self.silver_lakehouse_id}`.MarketingCampaignTaskRelatedParty mctrp 
                WHERE mctrp.SourceTable = 'actioneventinflow_relatedOrganization_IDM') mctrpo ON mct.TaskId = mctrpo.TaskId JOIN `{self.silver_lakehouse_id}`.MarketingCampaignTaskCommunication mctc ON mct.TaskId = mctc.TaskId 
                WHERE mct.MarketingCampaignTaskTypeId = 20000007"""
        source_df=spark.sql(source_query)
        target_df=spark.sql(target_query)
        source_count= source_df.collect()[0]["count"]
        target_count= target_df.collect()[0]["count"]
        self.assertEqual(source_count, target_count)
        
    def test_poa_compare_source_target_data_journeyevent(self):
        source_query="""Select count(*) as count FROM journeyeventinflow""" 
        target_query=f"""SELECT COUNT(*) AS count FROM `{self.silver_lakehouse_id}`.MarketingCampaignTask mct 
                            JOIN (SELECT rmct.TaskId, rmct.RelatedTaskId 
                                FROM `{self.silver_lakehouse_id}`.RelatedMarketingCampaignTask rmct 
                                WHERE rmct.SourceTable = 'journeyeventinflow_IDM') rmctj ON mct.TaskId = rmctj.TaskId JOIN `{self.silver_lakehouse_id}`.MarketingCampaignTaskStatus mcts ON mct.TaskId = mcts.TaskId 
                            JOIN `{self.silver_lakehouse_id}`.TaskStatusType tst 
                            ON mcts.TaskStatusTypeId = tst.TaskStatusTypeId 
                            JOIN (SELECT mctrp.PartyId, mctrp.TaskId 
                                FROM `{self.silver_lakehouse_id}`.MarketingCampaignTaskRelatedParty mctrp 
                                WHERE SourceTable = 'journeyeventinflow_IDM') mctrpt 
                            ON mct.TaskId = mctrpt.TaskId JOIN `{self.silver_lakehouse_id}`.Party pt 
                            ON mctrpt.PartyId = pt.PartyId JOIN (SELECT rmct.TaskId, rmct.RelatedTaskId 
                                FROM `{self.silver_lakehouse_id}`.RelatedMarketingCampaignTask rmct 
                                WHERE rmct.SourceTable = 'journeyeventinflow_relatedjourneyinstance_IDM') rmctji 
                            ON mct.TaskId = rmctji.TaskId JOIN (SELECT mctrp.PartyId, mctrp.TaskId 
                                FROM `{self.silver_lakehouse_id}`.MarketingCampaignTaskRelatedParty mctrp 
                                WHERE mctrp.SourceTable = 'journeyeventinflow_relatedOrganization_IDM') mctrpo 
                            ON mct.TaskId = mctrpo.TaskId WHERE mct.MarketingCampaignTaskTypeId = 20000003"""
        source_df=spark.sql(source_query)
        target_df=spark.sql(target_query)
        source_count= source_df.collect()[0]["count"]
        target_count= target_df.collect()[0]["count"]
        self.assertEqual(source_count, target_count)
        
    def test_poa_gold_tables_contains_data(self):
        table_names = ['JourneyDim','MarketingEventFact','MarketingChannelDim','MarketingAssetDim','AttributionModelDim']
        for table_name in table_names:
            count_df = spark.sql(f"SELECT COUNT(*) AS count FROM `{self.poagold_lakehouse_id}`.{table_name}")
            count = count_df.collect()[0]["count"]
            self.assertGreater(count, 0, "Each POA marketing table should have non-zero counts.")
        

def run_tests_and_write_output(spark):
    
    # Load and run tests
    loader = unittest.TestLoader()
    suite = loader.loadTestsFromTestCase(PatientOutreachAnalyticsDataValidationTests)

    # Inject parameters / context to the tests
    for test in suite:
        test.spark = spark
        test.workspace_id = workspace_id
        test.poagold_lakehouse_id=poagold_lakehouse_id
        test.silver_lakehouse_id=silver_lakehouse_id
    # Write XML test report to stream, decode after completion
    write_stream = io.BytesIO()
    xmlrunner.XMLTestRunner(output=write_stream, verbosity=3).run(suite)
    xml_output = write_stream.getvalue().decode('utf-8')

    # Write report to lakehouse
    mssparkutils.fs.put(f"abfss://{workspace_id}@{deployment_environment}-onelake.dfs.fabric.microsoft.com/{destination_lakehouse_id}/{destination_lakehouse_path}", xml_output, overwrite=True)
    return xml_output

report = run_tests_and_write_output(spark)